# 01 - Explore CIC-IDS2017

This notebook profiles the 55,726-flow working set (`data/intermediate/cicids2017_paper_base_glf.csv`) to mirror the descriptive figures and tables referenced in the paper.

**Outputs**
- Table 1: descriptive statistics for bytes, packets, and durations.
- Figure 1: log-scale histogram of bidirectional bytes.
- Figure 2: correlation heatmap for the paper features.
- Figure 3: missing-value visualization for the same fields.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-colorblind")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "intermediate" / "cicids2017_paper_base_glf.csv"
assert DATA_PATH.exists(), f"Missing dataset at {DATA_PATH}"

pd.set_option("display.float_format", "{:,.2f}".format)
DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print(f"Loaded {len(df):,} flows with {df.shape[1]} columns from {DATA_PATH.name}.")
df.head()

In [ ]:
profile_cols = [
    "bidirectional_bytes",
    "bidirectional_packets",
    "src2dst_packets",
    "dst2src_packets",
    "bidirectional_duration_ms",
]

table1 = (
    df[profile_cols]
    .apply(pd.to_numeric, errors="coerce")
    .describe(percentiles=[0.5, 0.9, 0.99])
    .T.rename(columns={"50%": "p50", "90%": "p90", "99%": "p99"})
)
table1[["count", "mean", "std", "min", "p50", "p90", "p99", "max"]]

In [ ]:
bytes_series = pd.to_numeric(df["bidirectional_bytes"], errors="coerce").fillna(0.0)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(bytes_series.clip(lower=1), bins=200, color="#4c72b0", alpha=0.85)
ax.set_xscale("log")
ax.set_xlabel("Bidirectional bytes (log scale)")
ax.set_ylabel("Flow count")
ax.set_title("Figure 1 – Distribution of bidirectional bytes (log)")
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.show()

In [ ]:
corr_cols = [
    "bidirectional_bytes",
    "bidirectional_packets",
    "src2dst_packets",
    "dst2src_packets",
    "bidirectional_duration_ms",
    "src2dst_first_seen_ms",
    "src2dst_last_seen_ms",
]

corr_df = df[corr_cols].apply(pd.to_numeric, errors="coerce")
corr = corr_df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
ax.set_title("Figure 2 – Spearman correlation matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Correlation")
plt.tight_layout()
plt.show()

In [ ]:
missing_pct = corr_df.isna().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(missing_pct)), missing_pct.values, color="#dd8452")
ax.set_xticks(range(len(missing_pct)))
ax.set_xticklabels(missing_pct.index, rotation=45, ha="right")
ax.set_ylabel("Missing share (%)")
ax.set_title("Figure 3 – Missingness for key features")
for idx, val in enumerate(missing_pct.values):
    ax.text(idx, val + 0.05, f"{val:.2f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

The flows show the expected heavy-tailed byte distribution with very small missing percentages in the features that feed the classifiers, matching the descriptive artifacts cited in the paper.